In [1]:
%load_ext autoreload

In [2]:
%autoreload 2
from utils import (
    select_users_by_period,
    create_hourly_user_dataset,
)
from visualization_utils import (
    plot_user_metrics,
    plot_market_features,
)


In [124]:
import pandas as pd
import numpy as np
import json
pd.set_option('display.max_columns', 500)

# MARKET = "eth_wbtc_usdc"
# MARKET = "eth_siusd_usdc"
MARKET = "eth_mapollo_usdc"

# MARKET = "base_cbbtc_usdc_full"
EVENTS_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_enriched"
HOURLY_MARKET_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_hourly_data"
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv(f"{EVENTS_PATH}/{MARKET}.csv")
market_df = pd.read_csv(f"{HOURLY_MARKET_PATH}/{MARKET}.csv")

df.head(3)

,hash,type,timestamp,user_address,assets,assets_usd,liquidated_assets,liquidated_assets_usd,market,datetime,market_address,total_supply_before,total_borrow_before,total_supply_after,total_borrow_after,utilization_before,utilization_after,tx_actions,borrow_rate_before,supply_rate_before,borrow_rate_after,supply_rate_after,collateral_price,loan_asset_price,collateral_before,collateral_value_before,debt_before,supply_before,ltv_before,collateral_after,collateral_value_after,debt_after,supply_after,ltv_after,health_factor_before,health_factor_after,event_type,vault_flg,volatility_6h,drawdown_6h,trend_6h,volatility_24h,drawdown_24h,trend_24h,event_sequence_type
0,0xdeab21215c1447f604843527533220f91ae5a73de161...,MarketSupply,1765379987,0x000000000000000000000000000000000000dEaD,1020,0.001020,0,0,eth_mapollo_usdc,2025-12-10 15:19:47,0x031c7333014af51e4fd18031d14e4eaada58348cde3f...,1.441375e+06,1.208459e+06,1.441386e+06,1.208470e+06,0.838407,0.838409,1,0.084369,0.070736,0.084369,0.070737,1.055091,0.999981,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.001020,0.0,0.0,0.0,loan_position_supply,False,0.000114,-0.000284,0.000285,0.000058,-0.000284,0.000285,loan_position_supply
1,0x0c4474276bf81ce7af82ca9041a836c48f42dd1c0268...,MarketSupply,1764793019,0x000001ac4e512d670c34feDf6c71cE2F49fb160a,50000000000,49990.165339,0,0,eth_mapollo_usdc,2025-12-03 20:16:59,0x031c7333014af51e4fd18031d14e4eaada58348cde3f...,1.109191e+06,9.175282e+05,1.159377e+06,9.177146e+05,0.827205,0.791558,1,0.091641,0.075807,0.088743,0.070246,1.053286,0.999803,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,49990.165339,0.0,0.0,0.0,loan_position_supply,False,0.000000,0.000000,0.000000,0.000075,-0.000360,0.000360,loan_position_supply
2,0xe37277632e2096024e7d8fd1d8c02b616c22749098ec...,MarketWithdraw,1765466951,0x000001ac4e512d670c34feDf6c71cE2F49fb160a,50078249384,50074.907520,0,0,eth_mapollo_usdc,2025-12-11 15:29:11,0x031c7333014af51e4fd18031d14e4eaada58348cde3f...,1.742913e+06,1.538826e+06,1.692856e+06,1.538847e+06,0.882905,0.909024,1,0.087513,0.077267,0.112812,0.102572,1.055416,0.999933,0.0,0.0,0.0,49996.663358,0.0,0.0,0.0,0.0,-78.244162,0.0,0.0,0.0,loan_position_withdraw,False,0.000123,-0.000308,0.000309,0.000063,-0.000308,0.000309,loan_position_withdraw


In [125]:
import pandas as pd
import numpy as np

def compute_hourly_hhi(events_df, hourly_df, n_hours=1, start_date=None):
    """
    Compute hourly supply concentration (HHI) from events.

    Parameters:
    events_df: DataFrame with columns 'user_address', 'timestamp', 'supply_after', 'collateral_value_after'
               (use supply_after for loan asset supply, or collateral_value_after for collateral supply)
    hourly_df: hourly market DataFrame with 'timestamp' and 'datetime'
    n_hours: step size in hours
    start_date: optional, only consider events after this date

    Returns:
    DataFrame with columns: timestamp, datetime, total_supply, hhi
    """
    df_events = events_df.sort_values(['timestamp', 'user_address', ]).copy()
    if start_date:
        start_ts = pd.Timestamp(start_date).timestamp()
        df_events = df_events[df_events['timestamp'] >= start_ts]
    
    # Get hourly timestamps from hourly_df (or generate from min/max)
    if hourly_df is not None:
        timestamps = hourly_df['timestamp'].values
    else:
        min_ts = df_events['timestamp'].min()
        max_ts = df_events['timestamp'].max()
        timestamps = np.arange(min_ts, max_ts + n_hours*3600, n_hours*3600)
    
    # Sort timestamps
    timestamps = np.sort(timestamps)
    
    # Track user supply state
    user_supply = {}
    result = []
    event_idx = 0
    events_ts = df_events['timestamp'].values
    events_supply = df_events['supply_after'].values  # use supply_after (USD)
    events_user = df_events['user_address'].values
    
    for ts in timestamps:
        # Update state with events that happened at or before ts
        while event_idx < len(events_ts) and events_ts[event_idx] <= ts:
            user = events_user[event_idx]
            new_supply = events_supply[event_idx]
            user_supply[user] = new_supply
            event_idx += 1        
        if not user_supply:
            # No supply yet
            result.append({'timestamp': ts, 'datetime': pd.to_datetime(ts, unit='s'),
                           'total_supply': 0, 'hhi': 0})
            continue

        
        total = sum(user_supply.values())
        n_suppliers = len(user_supply)
        supply_vals = np.array(list(user_supply.values()))
        if len(supply_vals) > 0:
            threshold = total * 0.05
            n_suppliers_above_5pct = (supply_vals >= threshold).sum()
        else:
            n_suppliers_above_5pct = 0
        if total == 0:
            result.append({'timestamp': ts, 'datetime': pd.to_datetime(ts, unit='s'),
                           'total_supply': 0, 'hhi': 0})
            continue
        
        # Calculate HHI
        hhi = sum((v / total) ** 2 for v in user_supply.values())
        result.append({'timestamp': ts, 'datetime': pd.to_datetime(ts, unit='s'),
                       'total_supply': total, 'hhi': hhi,
                       'n_suppliers': n_suppliers,
                       'n_suppliers_above_5pct': n_suppliers_above_5pct})
    
    return pd.DataFrame(result).fillna(0)
hhi_df = compute_hourly_hhi(
    df,
    market_df,
)
market_df["hhi"] = hhi_df["hhi"]
market_df["n_suppliers_above_5pct"] = hhi_df["n_suppliers_above_5pct"]

hhi_df

,timestamp,datetime,total_supply,hhi,n_suppliers,n_suppliers_above_5pct
0,1756976400,2025-09-04 09:00:00,0.000000e+00,0.000000,0.0,0.0
1,1756980000,2025-09-04 10:00:00,9.997120e-01,1.000000,2.0,1.0
2,1756983600,2025-09-04 11:00:00,9.997120e-01,1.000000,2.0,1.0
3,1756987200,2025-09-04 12:00:00,9.997120e-01,1.000000,2.0,1.0
4,1756990800,2025-09-04 13:00:00,9.997120e-01,1.000000,2.0,1.0
...,...,...,...,...,...,...
4411,1772856000,2026-03-07 04:00:00,1.821624e+06,0.790842,34.0,2.0
4412,1772859600,2026-03-07 05:00:00,1.818859e+06,0.792888,34.0,2.0
4413,1772863200,2026-03-07 06:00:00,1.810854e+06,0.798890,34.0,2.0
4414,1772866800,2026-03-07 07:00:00,1.810777e+06,0.798949,34.0,2.0


In [142]:
plot_market_features(
    market_df,
    ["total_supply", "utilization"]
)

In [131]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

def plot_supply_concentration(events_df, hourly_df=None, n_hours=1, start_date=None, threshold_pct=5):
    """
    Visualizes stacked supply volumes of major suppliers (left axis) and HHI (right axis).
    
    Parameters:
    events_df: DataFrame with columns 'user_address', 'timestamp', 'supply_after' (USD)
    hourly_df: optional hourly market DataFrame (used for timestamps)
    n_hours: step size in hours if hourly_df is not provided
    start_date: optional, only consider events after this date
    threshold_pct: minimum percentage of total supply for a user to be considered "major"
    """
    # Build hourly supply per user
    df_events = events_df.sort_values(['timestamp', 'user_address', ]).copy()
    if start_date:
        start_ts = pd.Timestamp(start_date).timestamp()
        df_events = df_events[df_events['timestamp'] >= start_ts]
    
    # Determine hourly timestamps
    if hourly_df is not None:
        timestamps = hourly_df['timestamp'].values
    else:
        min_ts = df_events['timestamp'].min()
        max_ts = df_events['timestamp'].max()
        timestamps = np.arange(min_ts, max_ts + n_hours*3600, n_hours*3600)
    timestamps = np.sort(timestamps)
    
    # Track user supply state
    user_supply = {}
    history = []  # list of dicts per timestamp
    event_idx = 0
    events_ts = df_events['timestamp'].values
    events_supply = df_events['supply_after'].values
    events_user = df_events['user_address'].values
    
    for ts in timestamps:
        while event_idx < len(events_ts) and events_ts[event_idx] <= ts:
            user = events_user[event_idx]
            new_supply = events_supply[event_idx]
            user_supply[user] = new_supply
            event_idx += 1
        
        if not user_supply:
            history.append({'timestamp': ts, 'users': {}, 'total': 0, 'hhi': 0})
            continue
        
        total = sum(user_supply.values())
        if total == 0:
            hhi = 0
        else:
            hhi = sum((v / total) ** 2 for v in user_supply.values())
        history.append({'timestamp': ts, 'users': user_supply.copy(), 'total': total, 'hhi': hhi})
    
    # Build DataFrame of supply per user per timestamp
    rows = []
    hhi_series = []
    for entry in history:
        ts = entry['timestamp']
        total = entry['total']
        hhi_series.append({'timestamp': ts, 'hhi': entry['hhi']})
        if total == 0:
            continue
        for user, supply in entry['users'].items():
            rows.append({'timestamp': ts, 'user': user, 'supply_usd': supply, 'total_supply': total})
    df_supply = pd.DataFrame(rows)
    df_hhi = pd.DataFrame(hhi_series)
    
    # Identify major suppliers
    if not df_supply.empty:
        df_supply['share'] = df_supply['supply_usd'] / df_supply['total_supply'] * 100
        major_users = df_supply.groupby('user')['share'].max()
        major_users = major_users[major_users > threshold_pct].index.tolist()
    else:
        major_users = []
    
    if not major_users:
        print(f"No user exceeds {threshold_pct}% of total supply at any time.")
        # Still plot HHI even if no major suppliers
        fig = go.Figure()
        x_vals = pd.to_datetime(df_hhi['timestamp'], unit='s')
        fig.add_trace(go.Scatter(x=x_vals, y=df_hhi['hhi'], mode='lines', name='HHI'))
        fig.update_layout(title='Supply Concentration (HHI) – No major suppliers found',
                          xaxis_title='Date', yaxis_title='HHI')
        fig.show()
        return fig
    
    # Pivot to get supply per user per timestamp, fill missing with 0
    pivot_supply = df_supply[df_supply['user'].isin(major_users)].pivot(index='timestamp', columns='user', values='supply_usd').fillna(0)
    all_ts = pd.Series(timestamps, name='timestamp')
    pivot_supply = pivot_supply.reindex(all_ts, fill_value=0)
    
    # Sort users by average supply for consistent stacking
    user_order = pivot_supply.mean().sort_values(ascending=False).index.tolist()
    pivot_supply = pivot_supply[user_order]
    
    # Convert timestamps to datetime for x-axis
    x_vals = pd.to_datetime(pivot_supply.index, unit='s')
    
    # Create figure with secondary y-axis
    fig = go.Figure()
    
    # Stacked area traces (left axis)
    cumulative = np.zeros(len(pivot_supply))
    colors = px.colors.qualitative.Plotly
    for i, user in enumerate(user_order):
        values = pivot_supply[user].values
        fig.add_trace(go.Scatter(
            x=x_vals,
            y=values,
            name=user[:10] + '...' if len(user) > 10 else user,
            stackgroup='one',
            mode='none',
            fill='tonexty',
            fillcolor=colors[i % len(colors)],
            line=dict(width=0),
            opacity=0.7,
            legendgroup='supply',
            showlegend=True
        ))
        cumulative += values
    
    # Total supply line (left axis)
    total_supply_series = pivot_supply.sum(axis=1).values
    fig.add_trace(go.Scatter(
        x=x_vals,
        y=total_supply_series,
        mode='lines',
        name='Total Supply',
        line=dict(color='black', width=2, dash='dash'),
        legendgroup='supply',
        showlegend=True
    ))
    
    # HHI line (right axis)
    # Merge HHI with the same timestamps
    df_hhi_merged = df_hhi.set_index('timestamp').reindex(pivot_supply.index).fillna(0)
    fig.add_trace(go.Scatter(
        x=x_vals,
        y=df_hhi_merged['hhi'].values,
        mode='lines',
        name='HHI (Supply Concentration)',
        line=dict(color='yellow', width=2),
        yaxis='y2'
    ))
    
    # Update layout with secondary y-axis
    fig.update_layout(
        title=f'Supply Concentration – Major Suppliers (> {threshold_pct}% of total)',
        xaxis_title='Date',
        yaxis_title='Supply Volume (USD)',
        yaxis2=dict(
            title='HHI',
            overlaying='y',
            side='right',
            showgrid=False,
            color='red'
        ),
        hovermode='x unified',
        legend_title='Legend'
    )
    fig.show()

cut_ts = market_df["timestamp"][len(market_df) // 3]


In [144]:
thr = market_df["total_supply"].max() * 0.01
print(thr)
cutoff_min_ts = market_df[market_df["total_supply"] > thr]["timestamp"].min()
cutoff_max_ts = market_df[market_df["total_supply"] > thr]["timestamp"].max()
cutoff_min_ts - cutoff_max_ts
df_prev = df[
    (df["timestamp"] > cutoff_min_ts) & 
    (df["timestamp"] < cutoff_max_ts)
    # (df["datetime"] < "2025-10-01")
]
market_df_prev = market_df[
    (market_df["timestamp"] > cutoff_min_ts) & 
    (market_df["timestamp"] < cutoff_max_ts)
    # (df["datetime"] < "2025-10-01")
]

plot_supply_concentration(
    df_prev,
    market_df_prev,
    threshold_pct=1
)

35109.677554033566


In [160]:
import pandas as pd

import numpy as np
import pandas as pd


import numpy as np
import pandas as pd

def detect_market_spikes(df, start_date, lookback_hours=6, metrics_config=None, actions_limit=10, max_recovery_events=50):
    """
    Detect spikes in specified market metrics and return enriched spike information.
    
    Returns list of dicts with keys:
        trigger_datetime, recovery_datetime (if recovered), recovery_time_seconds,
        spike_magnitudes (dict), trigger_event_types (comma-separated),
        market_state (dict with total_borrow, total_supply, collateral_price, loan_asset_price,
                     debt_before, supply_before, utilization_before),
        actions_df (DataFrame of actions during spike), spike_duration_events (int).
    """
    if metrics_config is None:
        metrics_config = {
            'utilization_after': {'spike_threshold': 0.05, 'high_threshold': 0.9, 'tolerance': 0.02}
        }
    
    if 'datetime' not in df.columns and 'timestamp' in df.columns:
        df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
    df = df.sort_values('timestamp').copy()
    
    start_ts = pd.Timestamp(start_date).timestamp()
    df = df[df['timestamp'] >= start_ts].reset_index(drop=True)
    if df.empty:
        return []
    
    timestamps = df['timestamp'].values
    lookback_sec = lookback_hours * 3600
    past_indices = np.searchsorted(timestamps, timestamps - lookback_sec, side='right') - 1
    
    metric_arrays = {}
    for metric in metrics_config.keys():
        if metric in df.columns:
            metric_arrays[metric] = df[metric].values
        else:
            raise KeyError(f"Metric '{metric}' not found in DataFrame columns")
    
    spikes = []
    i = 0
    n = len(df)
    while i < n and len(spikes) < actions_limit:
        past_idx = past_indices[i]
        if past_idx >= 0:
            triggered_metrics = {}
            baseline_vals = {}
            for metric, cfg in metrics_config.items():
                current_val = metric_arrays[metric][i]
                past_val = metric_arrays[metric][past_idx]
                delta = current_val - past_val
                if delta > cfg['spike_threshold'] and current_val > cfg['high_threshold'] and past_val < cfg['high_threshold']:
                    triggered_metrics[metric] = delta
                    baseline_vals[metric] = past_val
            
            if triggered_metrics:
                trigger_idx = i
                trigger_row = df.iloc[trigger_idx]
                trigger_tx_hash = trigger_row['hash']
                # Get all rows with same hash (same transaction) at trigger time
                trigger_tx_rows = df[(df['timestamp'] == trigger_row['timestamp']) & (df['hash'] == trigger_tx_hash)]
                trigger_event_types = ','.join(sorted(trigger_tx_rows['type'].unique()))
                
                # Market state at trigger (use before values or after? using before for consistency)
                market_state = {
                    'total_borrow': trigger_row.get('total_borrow_before', np.nan),
                    'total_supply': trigger_row.get('total_supply_before', np.nan),
                    'collateral_price': trigger_row.get('collateral_price', np.nan),
                    'loan_asset_price': trigger_row.get('loan_asset_price', np.nan),
                    'debt_before': trigger_row.get('debt_before', np.nan),
                    'supply_before': trigger_row.get('supply_before', np.nan),
                    'utilization_before': trigger_row.get('utilization_before', np.nan)
                }
                
                # Find recovery
                recovery_idx = None
                recovery_time = None
                for offset in range(max_recovery_events):
                    j = trigger_idx + offset
                    if j >= n:
                        break
                    all_recovered = True
                    for metric, cfg in metrics_config.items():
                        if metric in triggered_metrics:
                            current = metric_arrays[metric][j]
                            baseline = baseline_vals[metric]
                            # Recovery if: back to baseline (± tolerance) OR below high_threshold - tolerance
                            high_thresh = cfg['high_threshold']
                            low_threshold = cfg.get('low_threshold', None)
                            recovered = (abs(current - baseline) <= cfg['tolerance']) or (low_threshold is not None and current < low_threshold)
                            if not recovered:
                                all_recovered = False
                                break
                    if all_recovered:
                        recovery_idx = j
                        recovery_time = df.iloc[recovery_idx]['timestamp'] - df.iloc[trigger_idx]['timestamp']
                        break
                
                if recovery_idx is not None:
                    spike_df = df.iloc[trigger_idx:recovery_idx+1].copy()
                    next_i = recovery_idx + 1
                else:
                    # No recovery within limit, take up to max_recovery_events rows
                    end_idx = min(trigger_idx + max_recovery_events, n)
                    spike_df = df.iloc[trigger_idx:end_idx].copy()
                    recovery_time = None
                    next_i = end_idx
                
                spikes.append({
                    'trigger_datetime': trigger_row['datetime'],
                    'recovery_datetime': df.iloc[recovery_idx]['datetime'] if recovery_idx is not None else None,
                    'recovery_time_seconds': recovery_time,
                    'spike_magnitudes': triggered_metrics,
                    'trigger_event_types': trigger_event_types,
                    'market_state': market_state,
                    'actions_df': spike_df,
                    'spike_duration_events': len(spike_df)
                })
                i = next_i
                continue
        i += 1
    
    return spikes


metrics_config = {
    'utilization_after': {
        'spike_threshold': 0.05,    # 5% increase
        'high_threshold': 0.9,      # must exceed 90% utilization
        'low_threshold': 0.895,
        'tolerance': 0.01           # recovery within 2% of baseline
    },
    # 'borrow_rate_after': {
    #     'spike_threshold': 0.03,    # 1% increase
    #     'high_threshold': 0.08,      # must exceed 10% borrow rate
    #     'tolerance': 0.01          # recovery within 0.5% of baseline
    # }
}

spikes = detect_market_spikes(
    df_prev,
    df_prev["datetime"].min(),
    lookback_hours=1,
    metrics_config=metrics_config,
    actions_limit=1000,
)
len(spikes)

11

In [161]:
for i in spikes[:1550]:
    df_actions = i['actions_df'].copy()
    if df_actions.empty:
        continue
    
    trigger_ts = i['trigger_datetime']
    # df_actions = df_actions[df_actions['datetime'] > trigger_ts]
    if df_actions.empty:
        continue
    display(df_actions[[
        "datetime",
        "type",
        "utilization_before",
        "utilization_after",
        "borrow_rate_after",
        "user_address"
    ]])
    # if 'MarketRepay' in df_actions["type"].unique():
    #     display(df_actions[df_actions['type'] == 'MarketRepay'][[
    #         "datetime",
    #         "type",
    #         "utilization_before",
    #         "utilization_after",
    #         "borrow_rate_after",
    #         "user_address"
    #     ]])
        
    print("=" * 250)


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
50,2025-10-27 15:49:11,MarketWithdraw,0.620032,0.924409,0.110664,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78
51,2025-10-28 22:07:59,MarketSupply,0.924409,0.921223,0.108890,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78
52,2025-10-29 13:29:59,MarketSupply,0.921223,0.907241,0.082468,0x1a0C96eF4c775697D9ba9A99124dF73536Fd3866
53,2025-10-29 15:43:23,MarketSupplyCollateral,0.907241,0.996929,0.265087,0x6651dF73bE858037E7cFa8AfC82A33720b734f1f
54,2025-10-29 15:43:23,MarketBorrow,0.907241,0.996929,0.265087,0x6651dF73bE858037E7cFa8AfC82A33720b734f1f
55,2025-10-29 15:49:47,MarketRepay,0.996929,0.949726,0.169027,0x6651dF73bE858037E7cFa8AfC82A33720b734f1f
56,2025-10-29 20:10:23,MarketSupply,0.949726,0.906992,0.083114,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78
57,2025-10-29 20:49:47,MarketSupplyCollateral,0.906992,0.970097,0.213180,0x6651dF73bE858037E7cFa8AfC82A33720b734f1f
58,2025-10-29 20:49:47,MarketBorrow,0.906992,0.970097,0.213180,0x6651dF73bE858037E7cFa8AfC82A33720b734f1f
59,2025-10-29 20:57:11,MarketSupplyCollateral,0.970097,0.999999,0.274811,0x440Ecb31ee02460C1C1aA9d9b75476Cbfb4a3bD4


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
87,2025-11-05 08:10:47,MarketBorrow,0.894453,0.932857,0.171086,0x440Ecb31ee02460C1C1aA9d9b75476Cbfb4a3bD4
88,2025-11-05 08:10:47,MarketSupplyCollateral,0.894453,0.932857,0.171086,0x440Ecb31ee02460C1C1aA9d9b75476Cbfb4a3bD4
89,2025-11-05 08:13:35,MarketBorrow,0.932857,0.956859,0.233126,0x440Ecb31ee02460C1C1aA9d9b75476Cbfb4a3bD4
90,2025-11-05 08:13:35,MarketSupplyCollateral,0.932857,0.956859,0.233126,0x440Ecb31ee02460C1C1aA9d9b75476Cbfb4a3bD4
91,2025-11-05 08:30:35,MarketSupply,0.956859,0.919362,0.136205,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78
92,2025-11-05 14:53:59,MarketWithdraw,0.919362,0.919453,0.137408,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78
93,2025-11-05 18:39:59,MarketSupply,0.919453,0.911103,0.116186,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78
94,2025-11-05 21:40:47,MarketSupply,0.911103,0.894782,0.086969,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
95,2025-11-06 18:33:23,MarketWithdraw,0.894782,0.999999,0.349308,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78
96,2025-11-07 03:57:35,MarketWithdrawCollateral,0.999999,0.995731,0.354815,0xBD6f0bF29642c05613b10Afe99c4c465d2eAa37d
97,2025-11-07 03:57:35,MarketRepay,0.999999,0.995731,0.354815,0xBD6f0bF29642c05613b10Afe99c4c465d2eAa37d
98,2025-11-07 04:28:59,MarketWithdraw,0.995731,0.999999,0.368641,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78
99,2025-11-07 09:10:11,MarketRepay,0.999999,0.924294,0.163923,0x440Ecb31ee02460C1C1aA9d9b75476Cbfb4a3bD4
100,2025-11-07 09:11:59,MarketWithdrawCollateral,0.924294,0.924294,0.163923,0x440Ecb31ee02460C1C1aA9d9b75476Cbfb4a3bD4
101,2025-11-07 09:18:23,MarketRepay,0.924294,0.823991,0.088812,0x6651dF73bE858037E7cFa8AfC82A33720b734f1f


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
132,2025-11-13 20:39:59,MarketBorrow,0.891036,0.934691,0.203488,0x440Ecb31ee02460C1C1aA9d9b75476Cbfb4a3bD4
133,2025-11-14 08:46:59,MarketRepay,0.934691,0.891037,0.101198,0x6651dF73bE858037E7cFa8AfC82A33720b734f1f


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
171,2025-11-25 08:54:11,MarketBorrow,0.864350,0.915156,0.145362,0x440Ecb31ee02460C1C1aA9d9b75476Cbfb4a3bD4
172,2025-11-25 09:37:11,MarketRepay,0.915156,0.909893,0.129571,0x440Ecb31ee02460C1C1aA9d9b75476Cbfb4a3bD4
173,2025-11-25 17:06:35,MarketSupply,0.909893,0.900723,0.102564,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78
174,2025-11-25 22:15:59,MarketRepay,0.900723,0.890656,0.099631,0x440Ecb31ee02460C1C1aA9d9b75476Cbfb4a3bD4


...

,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
240,2025-12-11 10:02:23,MarketSupplyCollateral,0.784064,0.998157,0.34678,0x37829FE9b8e67B8267C2058b9459f524b9E3ca5d
241,2025-12-11 10:02:23,MarketBorrow,0.784064,0.998157,0.34678,0x37829FE9b8e67B8267C2058b9459f524b9E3ca5d
242,2025-12-11 11:51:23,MarketSupply,0.998157,0.883414,0.08716,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
279,2025-12-30 15:27:47,MarketWithdraw,0.801208,0.900685,0.065734,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78
280,2025-12-31 12:19:35,MarketWithdraw,0.900685,0.907582,0.079103,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78
281,2026-01-06 04:19:35,MarketWithdrawCollateral,0.907582,0.905430,0.079481,0x7915F95C6084e5cd3f7985D5609426372286BA28
282,2026-01-06 04:19:35,MarketRepay,0.907582,0.905430,0.079481,0x7915F95C6084e5cd3f7985D5609426372286BA28
283,2026-01-06 04:23:59,MarketRepay,0.905430,0.902792,0.074071,0x7915F95C6084e5cd3f7985D5609426372286BA28
284,2026-01-06 04:23:59,MarketWithdrawCollateral,0.905430,0.902792,0.074071,0x7915F95C6084e5cd3f7985D5609426372286BA28
285,2026-01-06 04:26:59,MarketRepay,0.902792,0.899274,0.068306,0x7915F95C6084e5cd3f7985D5609426372286BA28
286,2026-01-06 04:26:59,MarketWithdrawCollateral,0.902792,0.899274,0.068306,0x7915F95C6084e5cd3f7985D5609426372286BA28
287,2026-01-06 04:31:23,MarketRepay,0.899274,0.895316,0.068080,0x7915F95C6084e5cd3f7985D5609426372286BA28
288,2026-01-06 04:31:23,MarketWithdrawCollateral,0.899274,0.895316,0.068080,0x7915F95C6084e5cd3f7985D5609426372286BA28


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
317,2026-01-10 19:26:47,MarketWithdraw,0.834404,0.902385,0.070442,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78
318,2026-01-10 21:36:11,MarketBorrow,0.902385,0.902387,0.070447,0xe6db30dA89E8C6492dA3677fA9B5a7D59124995E
319,2026-01-11 20:25:35,MarketBorrow,0.902387,0.904232,0.074320,0xe6db30dA89E8C6492dA3677fA9B5a7D59124995E
320,2026-01-11 20:25:35,MarketSupplyCollateral,0.902387,0.904232,0.074320,0xe6db30dA89E8C6492dA3677fA9B5a7D59124995E
321,2026-01-14 11:48:35,MarketBorrow,0.904232,0.904439,0.075875,0x9F25E01155D7dfdde5653E42B60D0D015db9f3f3
322,2026-01-16 12:57:47,MarketWithdraw,0.904439,0.906156,0.080317,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78
323,2026-01-17 21:59:59,MarketSupply,0.906156,0.905501,0.079900,0x52a965C435CfFbc3EEbF9200550cC597F4E54029
324,2026-01-18 05:20:23,MarketWithdraw,0.905501,0.906188,0.081521,0x52a965C435CfFbc3EEbF9200550cC597F4E54029
325,2026-01-19 07:24:23,MarketSupply,0.906188,0.903158,0.075966,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78
326,2026-01-19 14:13:59,MarketSupply,0.903158,0.902753,0.075225,0x52a965C435CfFbc3EEbF9200550cC597F4E54029


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
426,2026-02-09 11:39:59,MarketWithdraw,0.553076,0.900001,0.077597,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78
427,2026-02-09 12:05:59,MarketSupply,0.900001,0.856236,0.074725,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
510,2026-03-02 04:25:47,MarketWithdraw,0.754135,0.993279,0.240571,0x7D91B200f2f59dFbED20Aad64E37719a1e805372
511,2026-03-02 04:32:11,MarketSupply,0.993279,0.848059,0.060594,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78


In [140]:
df[df["user_address"] == "0x6651dF73bE858037E7cFa8AfC82A33720b734f1f"][[
    "datetime",
    "type",
    "collateral_value_after",
    "debt_after",
    "ltv_after",
]].sort_values("datetime")
# df["collateral_price"]

,datetime,type,collateral_value_after,debt_after,ltv_after
280,2025-10-27 12:34:35,MarketSupplyCollateral,41611.618800,9998.966357,0.240317
281,2025-10-27 12:34:35,MarketBorrow,41611.618800,9998.966357,0.240317
282,2025-10-27 12:36:23,MarketSupplyCollateral,62417.428200,29996.899071,0.480635
283,2025-10-27 12:36:23,MarketBorrow,62417.428200,29996.899071,0.480635
284,2025-10-29 15:43:23,MarketSupplyCollateral,185734.820877,124987.054193,0.673003
285,2025-10-29 15:43:23,MarketBorrow,185734.820877,124987.054193,0.673003
286,2025-10-29 15:49:47,MarketRepay,185734.820877,74992.232516,0.403802
287,2025-10-29 20:49:47,MarketBorrow,216984.682832,144972.529930,0.668250
288,2025-10-29 20:49:47,MarketSupplyCollateral,216984.682832,144972.529930,0.668250
289,2025-11-03 15:56:59,MarketBorrow,NaN,154969.594172,0.000000


In [141]:
plot_user_metrics(
    df_prev,
    ["debt", "borrow_rate"],
    "0x6651dF73bE858037E7cFa8AfC82A33720b734f1f",
)

User address {'0x6651dF73bE858037E7cFa8AfC82A33720b734f1f'}
MAX DEBT {167750.16786031137}
0.0
167750.16786031137
